# Calculate OSDMA8: Highest seasonal (6-month) average of 8-hour daily maximum ozone concentrations across 15 months (Jan-Mar)

Input data: monthly mean MDA8 coordinates: lat, lon, time (month)

Output data: yearly values of OSDMA8 dimensions: lat, lon, year

In [ ]:
import os
import xarray as xr
from utils.utils import get_scenario_config

In [ ]:
# === Processing function ===
def calculate_maximum_6month_mean(start_year, end_year, monthly_mda8):
    years = list(range(start_year, end_year + 1))

    max_vals = []

    for year in years:
        # Define window: Jan of one year to Mar of the next year
        start = f"{year}-01"
        end = f"{year + 1}-03"

        # Subset to this window
        subset = monthly_mda8.sel(time=slice(start, end))

        # Compute 6-month rolling mean along time
        rolling_6m = subset.rolling(time=6, center=False).mean()

        # Find index of maximum
        max_idx = rolling_6m.argmax(dim="time")
        max_val = rolling_6m.isel(time=max_idx)

        # Expand dimensions for consistent output
        max_val = max_val.expand_dims(year=[year])

        max_vals.append(max_val)

    # Combine across years
    annual_max_6m = xr.concat(max_vals, dim="year")

    return annual_max_6m

In [ ]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "hist"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]

MDA8_DIR = f"/glade/work/awells/air_quality/{model}/ozone/MDA8/"
SAVE_DIR = f"/glade/work/awells/air_quality/{model}/ozone/OSDMA8/"

# === Main loop ===
for ens_num in ensemble_members:
    print(f"Processing {scenario}, Ensemble {ens_num:02d}")

    dates = f"{years.start}01-{years.stop}12"

    in_file = f"MDA8_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    file_path = os.path.join(MDA8_DIR, in_file)

    print(f"Reading {file_path}")
    monthly_mda8 = xr.open_dataarray(file_path)

    # Create list of years to calculate over
    start_year = int(str(monthly_mda8.time.dt.year[0].values))
    # Take second to last year to keep final March
    end_year = int(str(monthly_mda8.time.dt.year[-1].values - 1))

    annual_max_6m = calculate_maximum_6month_mean(start_year, end_year,
                                                  monthly_mda8)

    # Remove the unused time dimension, dimensions are now: lat, lon, year
    annual_max_6m = annual_max_6m.drop_vars("time")

    # Create date stamp for file saving
    start_time = annual_max_6m["year"][0].item()
    end_time = annual_max_6m["year"][-1].item()
    new_dates = f"{start_time}-{end_time}"

    out_file = f"OSDMA8_{model}_{scenario}_{ens_num:02d}_{new_dates}.nc"
    out_path = os.path.join(SAVE_DIR, out_file)

    print(f"Saving to {out_path}")
    description = ("OSDMA8: Highest seasonal (6-month) average of "
                   "8-hour daily maximum ozone concentrations across "
                   "15 months (Jan-Mar) - scripts by A.F. Wells (2025)")
    annual_max_6m.attrs["units"] = "ppb"
    annual_max_6m.attrs["description"] = description
    annual_max_6m.attrs["ensemble_number"] = ens_num
    annual_max_6m.attrs["scenario"] = scenario
    annual_max_6m.attrs["model"] = model
    annual_max_6m.to_netcdf(out_path)

print("All processing complete.")